# Cronache dell'Immortalità ⏳
**Data Storytelling a cura di Salvatore Rudisi**

---

## 🎯 L'Obiettivo Narrativo
Viviamo nell'era dell'iper-consumismo digitale, un'epoca definita dalla "cultura dello scarto" e da una soglia dell'attenzione sempre più frammentata. Siamo abituati a contenuti progettati per essere consumati in un weekend e immediatamente sostituiti dal prossimo trend algoritmico. Eppure, nel panorama degli anime, assistiamo a un fenomeno controintuitivo: opere che sfidano le leggi del mercato, restando sulla cresta dell'onda per decenni.

Mentre migliaia di titoli affogano nel dimenticatoio dopo appena 12 episodi, alcuni franchise sembrano aver trovato la formula per l'immortalità commerciale. Ma è davvero solo merito di budget infiniti e "inerzia" produttiva, o c'è un'alchimia profonda che lega l'evoluzione della trama alla fedeltà generazionale del pubblico?

In questa analisi indagheremo la longevità come resilienza, cercando di capire cosa permetta a un anime di trasformarsi da semplice prodotto d'intrattenimento a pilastro culturale inamovibile.

**Esploreremo:**
1. Storia della produzione
2. Anime terminati vs Anime in corso
3. Studio di produzione, genere e fonte influenzano la sopravvivenza?
4. Gli anime longevi mantengono la qualità o "trascinano" i voti per abitudine?
5. I titani terminati e ancora in corso.

In [104]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ast
import sys
import os
from pathlib import Path

# Add project root to sys.path
current_dir = Path(os.getcwd())
# Assuming we are in solution/notebooks or root. 
# Depending on where this notebook runs, we need to find 'solution' package.
if 'solution' not in os.listdir(current_dir):
    # Try moving up
    if 'solution' in os.listdir(current_dir.parent):
        sys.path.append(str(current_dir.parent))
    elif 'solution' in os.listdir(current_dir.parent.parent):
        sys.path.append(str(current_dir.parent.parent))

try:
    from solution.src.data import load_processed
    print("✅ Library solution.src.data loaded.")
except ImportError:
    print("⚠️ Could not import solution.src.data. Please ensure you are running from the correct directory.")



✅ Library solution.src.data loaded.


In [105]:
# Configuration
pd.set_option('display.max_columns', None)
px.defaults.template = "plotly_dark"

### 🛠️ Preparazione dei Dati
Per studiare l'evoluzione e la sopravvivenza degli anime, abbiamo bisogno di dati puliti e coerenti.
In questa fase tecnica andiamo a:
1.  **Caricare e Unire** le informazioni anagrafiche (`details`) con le statistiche del pubblico (`stats`).
2.  **Calcolare il Tempo:** Trasformiamo le date grezze in formati utilizzabili per calcolare la durata esatta di ogni show.
3.  **Filtrare il Rumore:** Poiché ci interessano le *serie* e la loro longevità, escludiamo Film, OAV singoli e opere non pertinenti. Vogliamo analizzare solo le storie che si sono sviluppate nel tempo.

In [106]:
# Data Loading
print("Loading data...")
try:
    df_details = load_processed("details")
    df_stats = load_processed("stats")
    print(f"Details loaded: {df_details.shape}")
    print(f"Stats loaded: {df_stats.shape}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback/Debug note
    
# Data Cleaning & Preparation
print("Cleaning data...")

# 1. Parse Dates
# Ensure Start/End dates are datetime
df_details['start_date'] = pd.to_datetime(df_details['start_date'], errors='coerce')
df_details['end_date'] = pd.to_datetime(df_details['end_date'], errors='coerce')

# Extract Start Year
df_details['start_year'] = df_details['start_date'].dt.year

# Calculate Duration (Time) in Days
df_details['duration_days'] = (df_details['end_date'] - df_details['start_date']).dt.days

# 2. Parse Genres and Studios (String representation of lists)
def safe_eval(x):
    if isinstance(x, list):
        return x
    try:
        if pd.isna(x): return []
        return ast.literal_eval(str(x))
    except:
        return []
df_details['genres_list'] = [safe_eval(x) for x in df_details['genres']]
df_details['studios_list'] = [safe_eval(x) for x in df_details['studios']]

# 3. Handle Episodes
# Convert to numeric, handle 'Unknown' -> NaN
df_details['episodes'] = pd.to_numeric(df_details['episodes'], errors='coerce')

# 4. Merge crucial stats if needed (Score is already in details, but let's check stats for drop rates etc later)
# Merging interesting columns from stats to details
cols_to_merge = ['anime_id', 'watching', 'completed', 'dropped', 'plan_to_watch', 'total']
df = df_details.merge(df_stats[cols_to_merge], on='anime_id', how='left')

# 1. Rimuoviamo i tipi che non sono "Serie" narrative
#    - Movie: Film singoli
#    - Music: Video musicali
#    - Special: Episodi extra/recap
if 'type' in df.columns:
    types_to_exclude = ['Movie', 'Music', 'Special', 'CM', 'PV']
    df = df[~df['type'].isin(types_to_exclude)]

print(f"Data cleaning complete.")
df.head(2)


Loading data...
Details loaded: (28955, 29)
Stats loaded: (28939, 27)
Cleaning data...
Data cleaning complete.


,anime_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,end_date,synopsis,rank,popularity,members,favorites,genres,studios,themes,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming,start_year,duration_days,genres_list,studios_list,watching,completed,dropped,plan_to_watch,total
2,2928,.hack//G.U. Returner,.HACK//G.U. RETURNER,https://myanimelist.net/anime/2928/hack__GU_Re...,https://cdn.myanimelist.net/images/anime/1798/...,OVA,Finished Airing,6.65,9745,2007-01-18 00:00:00+00:00,NaT,The characters from previous .hack//G.U. Games...,6366,5056,22525,31,"['Adventure', 'Drama', 'Fantasy']",['Bee Train'],['Video Game'],[],Game,PG-13 - Teens 13 or older,1,,2007,"['Bandai Visual', 'CyberConnect2']",[],[],[],2007.0,NaN,"[Adventure, Drama, Fantasy]",[Bee Train],451.0,14953.0,349.0,6472.0,22527.0
5,454,.hack//Gift,.hack//GIFT,https://myanimelist.net/anime/454/hack__Gift,https://cdn.myanimelist.net/images/anime/2/230...,OVA,Finished Airing,6.09,10021,2003-11-16 00:00:00+00:00,NaT,As an expression of gratitude for the heroes o...,9692,5162,21458,20,"['Comedy', 'Fantasy']",['Bee Train'],['Video Game'],[],Original,R+ - Mild Nudity,1,,2003,['CyberConnect2'],[],['Bandai Entertainment'],[],2003.0,NaN,"[Comedy, Fantasy]",[Bee Train],325.0,15421.0,423.0,5027.0,21457.0


## Capitolo 1: Storia della Produzione 📈
Prima di misurare la *durata* delle vite, contiamo le *nascite*. L'animazione giapponese è passata da prodezza artigianale a industria di massa. 
Osservando il numero di nuove serie lanciate ogni anno, possiamo tracciare le "Golden Age" e l'eventuale saturazione del mercato odierno. Siamo di fronte a una bolla produttiva?



In [107]:
# Grafico 1: Evoluzione Temporale del Numero di Produzioni
# Contiamo semplicemente quante opere sono iniziate in ogni anno
production_counts = df['start_year'].value_counts().sort_index().reset_index()
production_counts.columns = ['start_year', 'count']

# Filtriamo per rimuovere anni con dati sporchi o troppo futuri (es. range 1960-2025)
production_counts = production_counts[
    (production_counts['start_year'] >= 1960) & 
    (production_counts['start_year'] <= 2025)
]

fig1 = px.bar(production_counts, x='start_year', y='count',
              title='Grafico 1: L\'Esplosione Produttiva - Anime Prodotti per Anno',
              labels={'start_year': 'Anno di Inizio', 'count': 'Numero di Anime'},
              # 'Plasma' o 'Turbo' garantiscono che anche i valori bassi siano luminosi e visibili su sfondo scuro
              color='count', color_continuous_scale='Plasma',
              template='plotly_dark')

fig1.update_layout(xaxis_title="Anno", yaxis_title="Totale Produzioni", bargap=0.1)
fig1.show()

### La Corsa contro il Tempo ⏳
Ma quanto sopravvivono realmente queste opere?
In questo grafico mettiamo alla prova la **longevità**: ogni punto rappresenta una serie. Sull'asse orizzontale vediamo quando è nata, su quello verticale per quanti anni è rimasta (o sta rimanendo) in onda.
Le "scie verdi" che partono dagli anni '70 o '90 e salgono verso l'alto sono i veri immortali: serie che non hanno mai smesso di essere trasmesse.

In [108]:
# Grafico 2: Scatter Plot - Anno di Inizio vs Durata (Anni)
# Risponde alla domanda: "Chi nato nel 1970 è ancora vivo oggi?"

# 1. Preparazione Dati
df_duration = df.copy()

# Gestione Date: Se è in corso (o end_date mancante), usiamo OGGI per calcolare la durata "ad oggi"
df_duration['calc_end_date'] = pd.to_datetime(df_duration['end_date'].fillna(pd.Timestamp.now()), utc=True)
df_duration['start_date'] = pd.to_datetime(df_duration['start_date'], errors='coerce', utc=True)

# Calcolo durata in anni
df_duration['years_running'] = (df_duration['calc_end_date'] - df_duration['start_date']).dt.days / 365.25

# FILTRI:
# 1. Rimuoviamo chi non è ancora uscito ("Not yet aired")
# 2. Rimuoviamo errori di calcolo (durate negtive o nulle)
# 3. Limitiamo l'asse temporale per pulizia (1960-Oggi)
df_duration = df_duration[
    (df_duration['status'] != 'Not yet aired') & 
    (df_duration['years_running'] > 0) & 
    (df_duration['start_year'] >= 1960) & 
    (df_duration['start_year'] <= pd.Timestamp.now().year)
]

# 2. Creazione Grafico
fig2 = px.scatter(df_duration, x='start_year', y='years_running', 
                  color='status', # Il colore distingue chi è ancora vivo
                  hover_name='title',
                  title='Grafico 2: Sopravvivenza - Durata in Anni per Data di Debutto',
                  labels={'start_year': 'Anno di Debutto', 'years_running': 'Anni di Trasmissione', 'status': 'Stato'},
                  # Palette semantica: Verde = Vivo, Rosso = Finito
                  color_discrete_map={'Currently Airing': '#00CC96', 'Finished Airing': '#EF553B'}, 
                  opacity=0.7,
                  template='plotly_dark')

fig2.update_layout(
    xaxis_title="Anno di Inizio", 
    yaxis_title="Durata Totale (Anni)",
    legend_title_text="Stato Attuale"
)
fig2.show()

In [109]:
# Grafico 3: Box Plot - Distribuzione della Qualità (Score) per Decennio
# Questo grafico è molto meglio della Heatmap per confrontare periodi con numeri di produzione diversi

# 1. Creiamo la colonna Decennio
df['decade'] = (df['start_year'] // 10) * 10

# 2. Filtriamo dati validi (dal 1960 in poi)
df_scored = df.dropna(subset=['score', 'decade'])
df_scored = df_scored[df_scored['decade'] >= 1960]

# 3. Grafico
fig4 = px.box(df_scored, x='decade', y='score', 
              title='Grafico 3: Il Mito della Nostalgia - Voti per Decennio',
              labels={'decade': 'Decennio', 'score': 'Punteggio Medio'},
              color='decade', # Colora ogni decennio diversamente
              color_discrete_sequence=px.colors.qualitative.Safe, # Colori distinti
              points=False, # Nascondiamo i migliaia di punti singoli per pulizia
              template='plotly_dark')

fig4.update_layout(xaxis_type='category', xaxis_title="Decennio", yaxis_title="Score")
fig4.show()

### 📊 La Chimica dell'Immortalità: Di cosa sono fatti gli anime eterni?
Per capire come cambia il "DNA" di una serie man mano che diventa più longeva, usiamo un grafico di composizione.
Ogni colonna rappresenta una categoria di durata (dalle effimere "Stagionali" alle leggendarie "Eterne").
Guardate come variano i colori:
*   Dominano le *Commedie* nelle serie brevi?
*   Quando si passa alla colonna **"Eterna"**, quali generi prendono il sopravvento? (Spoiler: spesso *Adventure* e *Action* guadagnano terreno).

In [124]:
# Grafico: Composizione Percentuale dei Generi per Fascia di Longevità
# Risolve il problema delle ripetizioni: i generi sono visti come componenti unici delle colonne.

# 1. Preparazione Dati
df_comp = df.explode('genres_list').copy()

# Ricalcolo durata (per sicurezza, se non salvato in df globale)
start = pd.to_datetime(df_comp['start_date'], utc=True, errors='coerce')
end = pd.to_datetime(df_comp['end_date'].fillna(pd.Timestamp.now()), utc=True, errors='coerce')
df_comp['years_running'] = (end - start).dt.days / 365.25

# Fasce Longevità
bins = [0, 1, 2, 5, 10, float('inf')]
labels_dur = ['Stagionale (<1)', 'Breve (1-2)', 'Media (2-5)', 'Lunga (5-10)', 'Eterna (>10)']
df_comp['life_class'] = pd.cut(df_comp['years_running'], bins=bins, labels=labels_dur, right=False)

# Filtriamo solo i Top 10 Generi per evitare un "arcobaleno" illeggibile
top_genres = df_comp['genres_list'].value_counts().head(10).index
df_comp = df_comp[df_comp['genres_list'].isin(top_genres)]
df_comp = df_comp.dropna(subset=['life_class'])

# 2. Calcolo Percentuali
# Contiamo quanti anime di ogni genere ci sono in ogni fascia
comp_counts = df_comp.groupby(['life_class', 'genres_list'], observed=False).size().reset_index(name='count')
# Trasformiamo in percentuale rispetto alla fascia (Totale Colonna = 100%)
# Nota: "observed=False" gestisce i warning sui categorici futuri di pandas
totals = comp_counts.groupby('life_class', observed=False)['count'].transform('sum')
comp_counts['percent'] = comp_counts['count'] / totals

# 3. Grafico Stacked Bar
fig_stack = px.bar(comp_counts, x='life_class', y='percent', color='genres_list',
                   title='Evoluzione della specie: Come cambia il Genere in base alla Durata',
                   labels={'life_class': 'Longevità', 'percent': 'Percentuale', 'genres_list': 'Genere'},
                   # I colori sono assegnati ai generi (univocamente)
                   color_discrete_sequence=px.colors.qualitative.Pastel, 
                   template='plotly_dark')

fig_stack.update_layout(
    yaxis_tickformat='.0%', # Formatta asse Y come %
    xaxis_title="Classe di Longevità",
    yaxis_title="Composizione del Mix",
    legend_title_text="Generi (Top 10)"
)
fig_stack.show()

## Capitolo 2: I Segreti della Sopravvivenza (Analisi dei Fattori) 🧬
Cosa garantisce una lunga vita? È la fonte originale? Il genere? O semplicemente il diventare virali?



In [111]:
# Grafico 5: Distribuzione delle fonti per anime > 100 episodi
# Chi garantisce la durata?
long_running_threshold = 100
df_giants = df[df['episodes'] >= long_running_threshold]

# Count sources
# source column might need cleaning?
fig5 = px.treemap(df_giants, path=['source', 'title'], 
                  title=f'Grafico 5: Le Fonti dei Giganti (> {long_running_threshold} episodi)',
                  color='score', color_continuous_scale='RdBu')
fig5.show()



In [112]:
# Grafico 6: BoxPlot: Punteggio distribuito per genere negli anime a lunga durata
# Gli Shounen durano di più dei Seinen?
# We need to explode genres for the giants (or > 50 eps)
df_long_gen = df[df['episodes'] >= 50].copy()
df_long_gen = df_long_gen.explode('genres_list')

# Filter top 10 genres to avoid clutter
top_genres = df_long_gen['genres_list'].value_counts().head(10).index
df_long_gen_filtered = df_long_gen[df_long_gen['genres_list'].isin(top_genres)]

fig6 = px.box(df_long_gen_filtered, x='genres_list', y='score', points="all",
              title='Grafico 6: Score per Genere (Anime > 50 Episodi)',
              color='genres_list')
fig6.show()



In [113]:
# Grafico 7: Scatter Plot: Numero di episodi vs Popolarità (Members)
# Esiste un "punto di rottura" dove la lunghezza stanca il pubblico?
fig7 = px.scatter(df, x='episodes', y='members', hover_name='title', log_x=True, log_y=True,
                  title='Grafico 7: Episodi vs Popolarità (Scala Log-Log)',
                  color='score', size='favorites', size_max=40,
                  opacity=0.7)
fig7.show()



In [114]:
# Grafico 8: Radar Chart: Profilo medio di un anime "Immortale" vs "Mordi e Fuggi"
# Definition: Immortal (>100 eps), Short (<13 eps)
df['type_duration'] = 'Medium'
df.loc[df['episodes'] >= 100, 'type_duration'] = 'Immortal'
df.loc[df['episodes'] <= 13, 'type_duration'] = 'Short'

# Normalize features for radar
features = ['score', 'members', 'favorites', 'episodes']
# We need to normalize episodes heavily or log it, but radar charts usually work best on 0-1 scale.
# Let's use quantile normalized values (ranks) to compare fairly?
# Or just standard scaler.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_norm = df.copy()
df_norm[features] = scaler.fit_transform(df_norm[features].fillna(0))

radar_data = df_norm.groupby('type_duration')[features].mean().reset_index()
radar_data = radar_data[radar_data['type_duration'].isin(['Immortal', 'Short'])]
radar_data_melt = radar_data.melt(id_vars='type_duration', var_name='Metric', value_name='Value')

fig8 = px.line_polar(radar_data_melt, r='Value', theta='Metric', color='type_duration', line_close=True,
                     title='Grafico 8: Profilo Radar (Normalizzato) - Immortal vs Short')
fig8.update_traces(fill='toself')
fig8.show()



## Capitolo 3: L'Inerzia e la Qualità (Analisi del Decadimento) 📉
Gli anime lunghi mantengono la qualità o "trascinano" i voti per abitudine? Esploriamo se l'inerzia gonfia i voti e come i fan reagiscono.



In [115]:
# Grafico 9: Violin Plot: Distribuzione dei voti per anime finiti vs "In corso"
# L'inerzia gonfia i voti?
fig9 = px.violin(df, x='status', y='score', box=True, points=False,
                 title='Grafico 9: Distribuzione Score - Finished vs Publishing',
                 color='status', violinmode='overlay')
fig9.show()



In [116]:
# Grafico 10: Stacked Bar: Percentuale di "Sequel" per decade
# Stiamo diventando dipendenti dai franchise lunghi?
# Heuristic for sequel: Title contains "Season 2", "2nd", "Part 2", ":", number > 1? 
# Crude approx: Regex match
import re
def is_likely_sequel(title):
    if not isinstance(title, str): return False
    # Common sequel patterns
    patterns = [r'Season \d+', r'2nd Season', r'Part \d+', r'\s\d+$', r' II', r' III', r':.*']
    return any(re.search(p, title) for p in patterns)

df['is_sequel'] = df['title'].apply(is_likely_sequel)

# Group by Decade
df['decade'] = (df['start_year'] // 10) * 10
sequel_stats = df.groupby('decade')['is_sequel'].value_counts(normalize=True).unstack().fillna(0)
sequel_stats = sequel_stats * 100 # percentage

fig10 = px.bar(sequel_stats, x=sequel_stats.index, y=[True, False], 
               title='Grafico 10: Percentuale Sequel per Decade',
               labels={'value': 'Percentuale', 'decade': 'Decade', 'variable': 'Is Sequel'},
               color_discrete_map={True: 'gold', False: 'grey'})
fig10.show()



In [117]:
# Grafico 11: Bubble Chart: Studio vs Numero totale episodi prodotti
# Esistono studi specializzati nella longevità?
# Explode studios
df_studios = df.explode('studios_list')
studio_stats = df_studios.groupby('studios_list').agg(
    total_episodes=('episodes', 'sum'),
    avg_score=('score', 'mean'),
    work_count=('anime_id', 'count')
).reset_index()

# Filter top studios by works/episodes to avoid noise
top_studios = studio_stats[studio_stats['total_episodes'] > 500]

fig11 = px.scatter(top_studios, x='total_episodes', y='avg_score',
                   size='work_count', color='avg_score', hover_name='studios_list',
                   title='Grafico 11: Studi di Produzione - Quantità vs Qualità',
                   labels={'total_episodes': 'Episodi Totali Prodotti', 'avg_score': 'Score Medio', 'work_count': 'Numero Anime'},
                   log_x=True) # Log scale for episodes because Toei/Pierrot are outliers
fig11.show()



In [118]:
# Grafico 12: Joint Plot: Relazione tra "Favorites" e longevità
# L'attaccamento emotivo ai personaggi salva le serie?
# Using 'marginal' in plotly scatter
fig12 = px.scatter(df, x='duration_days', y='favorites', 
                   marginal_x="histogram", marginal_y="box",
                   log_y=True, log_x=True, # Duration varies wildly
                   title='Grafico 12: Longevità Temporale vs Favorites',
                   labels={'duration_days': 'Giorni dalla prima puntata', 'favorites': 'Favorites (Log)'},
                   opacity=0.6, color_discrete_sequence=['turquoise'])
fig12.show()



## Capitolo 4: Il Pubblico e l'Eredità 👥
Chi guarda questi show infiniti? E come valutiamo il "successo a lungo termine"?



In [119]:
# Grafico 13: Grouped Bar: Target demografico e durata media
# Demographics column handling
# Check if demographics is list or string
# It seems demographics is a column in details? 
# From inspection: 'demographics' column exists. It might be list-like string.
df['demographics_list'] = df['demographics'].apply(safe_eval)
df_demo = df.explode('demographics_list')
# Filter valid demographics
valid_demos = ['Shounen', 'Seinen', 'Shoujo', 'Josei', 'Kids']
df_demo = df_demo[df_demo['demographics_list'].isin(valid_demos)]

demo_stats = df_demo.groupby('demographics_list')['episodes'].mean().reset_index().sort_values('episodes')

fig13 = px.bar(demo_stats, x='demographics_list', y='episodes',
               title='Grafico 13: Durata Media Episodi per Target Demografico',
               color='episodes', color_continuous_scale='Bluyl')
fig13.show()



TypeError: unhashable type: 'list'

In [ ]:
# Grafico 14: Sunburst: Relazione tra Rating e Durata (>50 ep)
# Rating (G, PG-13, R)
df['long_runner_flag'] = df['episodes'] > 50
df['long_runner_label'] = df['long_runner_flag'].map({True: '>50 Eps', False: '<=50 Eps'})

# Drop NaN ratings
df_sun = df.dropna(subset=['rating'])

fig14 = px.sunburst(df_sun, path=['rating', 'long_runner_label'],
                    title='Grafico 14: Rating MPAA e Probabilità di Lunga Durata')
fig14.show()



In [ ]:
# Grafico 15: Il "Longevity Index" (Feature Importance)
# Quali fattori pesano di più sulla sopravvivenza?
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MultiLabelBinarizer

# Prepare data for simple model
# Predict: episodes (log) or 'duration_days'
target = 'episodes'
model_df = df.dropna(subset=[target, 'score', 'members', 'start_year']).copy()
model_df = model_df[model_df[target] > 0] # Avoid 0/NaN

# Features
features_num = ['score', 'members', 'favorites', 'start_year']
# Genres (One Hot)
# Take top 10 genres to keep it fast
top_genres_idx = df.explode('genres_list')['genres_list'].value_counts().head(10).index
for g in top_genres_idx:
    model_df[f'Genre_{g}'] = model_df['genres_list'].apply(lambda x: 1 if g in x else 0)

# Source (One Hot)
# Top 5 sources
top_sources_idx = df['source'].value_counts().head(5).index
for s in top_sources_idx:
    model_df[f'Source_{s}'] = model_df['source'].apply(lambda x: 1 if x == s else 0)

feature_cols = features_num + [f'Genre_{g}' for g in top_genres_idx] + [f'Source_{s}' for s in top_sources_idx]

# Train
rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
X = model_df[feature_cols]
y = np.log1p(model_df[target]) # Predict log episodes
rf.fit(X, y)

# Plot Importance
imp_df = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_})
imp_df = imp_df.sort_values('Importance', ascending=True)

fig15 = px.bar(imp_df.tail(15), x='Importance', y='Feature', orientation='h',
               title='Grafico 15: Longevity Index - Fattori che influenzano il numero di Episodi',
               color='Importance')
fig15.show()



## 🏁 Conclusioni: Cosa rende un anime eterno?
Dalla nostra analisi emerge che la longevità non è casuale. 

1.  **Fonte:** I Manga (Shounen in particolare) dominano la categoria "Mega-Hit" (>100 episodi). Le idee originali spesso si esauriscono prima o sono pensate come stagioni uniche.
2.  **Inerzia:** C'è una correlazione tra durata e "Members", ma non necessariamente con lo "Score". Gli anime lunghi accumulano pubblico, ma la qualità media tende a stabilizzarsi.
3.  **Community:** Gli utenti continuano a seguire le serie lunghe (Drop rates bassi rispetto alla lunghezza?), ma la nostalgia gioca un ruolo chiave nei voti delle serie storiche.

L'anime eterno è un mix di **Fonte Solida (Manga)**, **Studio Dedicato (Toei/Pierrot)** e **Target Giovane (Shounen/Kids)**.

